In [1]:
# Project setup: paths and shared model utilities; run before analysis.
from pathlib import Path


def find_project_root():
    """Locate the parent of the Code and Data package folders."""
    current = Path.cwd().resolve()
    for base in (current, *current.parents):
        for candidate in (base, base / "my paper"):
            if ((candidate / "Energy Economics Code" / "matlab code").is_dir()
                and (candidate / "Energy Economics Data").is_dir()):
                return candidate.resolve()
    raise FileNotFoundError("Project root not found. Start the notebook with the kernel working "
        "directory inside the project folder containing Energy Economics Code "
        "and Energy Economics Data. See readme.txt.")

PROJECT_ROOT = find_project_root()
PACKAGE_ROOT = PROJECT_ROOT / "Energy Economics Code"
DATA_ROOT = PROJECT_ROOT / "Energy Economics Data"
RESILIENCE_FILE = PACKAGE_ROOT / "matlab code" / "resilience_metrics.xlsx"
MACRO_DATA_FILE = DATA_ROOT / "markov_scale_data_166.csv"
OUTPUT_ROOT = PACKAGE_ROOT / "python code" / "output"
NOTEBOOK_OUTPUT_DIR = OUTPUT_ROOT
OUTPUT_DIR = NOTEBOOK_OUTPUT_DIR

for label, input_file in (("Resilience workbook", RESILIENCE_FILE),
    ("166-observation conditioning data", MACRO_DATA_FILE),):
    if not input_file.is_file():
        raise FileNotFoundError(f"{label} not found: {input_file}")

print(f"Resilience input: {RESILIENCE_FILE}")
print(f"Conditioning input: {MACRO_DATA_FILE}")
print(f"Output directory: {OUTPUT_DIR}")

# Shared model utilities. Their numpy/pandas/scipy imports remain in Cells 3 and 4.


def get_parameter_names(result):
    """Return parameter names from a statsmodels result."""
    if hasattr(result.params, "index"):
        return list(result.params.index)

    return list(result.model.param_names)


def make_named_series(values, names):
    """Convert a parameter array into a named pandas Series."""
    return pd.Series(np.asarray(values, dtype=float), index=names)


def find_parameter(names, target):
    """Find a parameter using exact string matching."""
    for name in names:
        if str(name) == target:
            return name

    return None


def find_regime_parameter(names, variable, regime, exog_columns=None):
    """
    Find a regime-dependent parameter.

    This supports both possible statsmodels naming formats:
        DXY[0], GSCPI[1], ...
    and:
        x1[0], x2[1], ...
    """
    if variable == "const":
        candidates = [f"const[{regime}]"]
    else:
        candidates = [f"{variable}[{regime}]"]

        if exog_columns is not None and variable in exog_columns:
            variable_position = exog_columns.index(variable) + 1
            candidates.append(f"x{variable_position}[{regime}]")

    for candidate in candidates:
        parameter_name = find_parameter(names, candidate)

        if parameter_name is not None:
            return parameter_name

    raise ValueError(f"Cannot find a regime parameter for variable '{variable}', "
        f"regime {regime}.\n"
        f"Tried parameter names: {candidates}\n"
        f"Available parameters:\n{names}")


def significance_stars(p_value):
    """Return conventional significance stars."""
    if pd.isna(p_value):
        return ""

    if p_value < 0.01:
        return "***"

    if p_value < 0.05:
        return "**"

    if p_value < 0.10:
        return "*"

    return ""


def calculate_z_p(estimate, standard_error):
    """Calculate a z-statistic and its two-sided normal p-value."""
    if (pd.isna(estimate)
        or pd.isna(standard_error)
        or standard_error == 0):
        return np.nan, np.nan

    z_value = estimate / standard_error
    p_value = 2.0 * norm.sf(abs(z_value))

    return z_value, p_value


def clean_small_number(value, decimals=4):
    """Avoid displaying rounded values such as -0.0000."""
    if pd.isna(value):
        return value

    threshold = 0.5 * 10 ** (-decimals)

    if abs(value) < threshold:
        return 0.0

    return value


def paper_format(estimate, z_value, p_value):
    """
    Produce the format used in the LaTeX table:

        0.8251*** (313.762)
       -0.0031    (-1.247)
    """
    estimate = clean_small_number(estimate, decimals=4)
    z_value = clean_small_number(z_value, decimals=3)
    stars = significance_stars(p_value)

    return f"{estimate:.4f}{stars} ({z_value:.3f})"


def validate_uncertainty_selection(selection):
    """Validate and normalize the uncertainty-index selection."""
    normalized = str(selection).strip().upper()
    valid_options = set(uncertainty_variable_maps) | {"ALL"}

    if normalized not in valid_options:
        raise ValueError(f"SELECTED_UNCERTAINTY must be one of "
            f"{sorted(valid_options)}; received '{selection}'.")

    if normalized == "ALL":
        return list(uncertainty_variable_maps.keys())

    return [normalized]


def validate_variable_codes(r_data, uncertainty_name, variable_map):
    """Check that all required Var codes are present in the Excel file."""
    available_codes = set(r_data["Variable"].astype(str))
    missing_codes = [code
        for code in variable_map.values()
        if code not in available_codes]

    if missing_codes:
        raise ValueError(f"The following variable codes required for {uncertainty_name} "
            f"are missing from resilience_metrics.xlsx: {missing_codes}")


def build_dependent_dataframe(r_data, value_column, variable_map):
    """
    Build a five-market dependent-variable DataFrame.

    Parameters
    ----------
    value_column : str
        RP -> resilience intensity (RI)
        RD -> resilience duration (RD)
    variable_map : dict
        Market-to-Var-code mapping for GPR, CPU, or OPU.
    """
    market_series = {}

    for market in markets:
        variable_code = variable_map[market]

        market_series[market] = (pd.to_numeric(r_data.loc[
                    r_data["Variable"].astype(str) == variable_code, value_column,],
                errors="coerce")
            .reset_index(drop=True))

    return pd.DataFrame(market_series)


def relabel_regimes(measure_name, regime_results):
    """
    Relabel raw statsmodels regimes according to economic meaning.

    RI:
        State 1 = high absorption = larger conditional mean.
        State 2 = low absorption  = smaller conditional mean.

    RD:
        State 1 = fast recovery = smaller conditional mean.
        State 2 = slow recovery = larger conditional mean.
    """
    if measure_name == "RI":
        ordered_raw_regimes = sorted([0, 1],
            key=lambda regime: regime_results[regime]["conditional_mean"], reverse=True)
    elif measure_name == "RD":
        ordered_raw_regimes = sorted([0, 1],
            key=lambda regime: regime_results[regime]["conditional_mean"])
    else:
        raise ValueError(f"Unknown resilience measure: {measure_name}")

    return ordered_raw_regimes[0], ordered_raw_regimes[1]


def paper_display_columns():
    """Return a fresh list of the existing compact-table columns."""
    return ["Market",
        "State 1 Cond. mean", "State 1 DXY", "State 1 GSCPI", "State 1 CPI", "State 1 Industrial",
        "State 2 Cond. mean", "State 2 DXY", "State 2 GSCPI", "State 2 CPI", "State 2 Industrial",
        "p11", "p22"]


def print_section(title, separator):
    """Print the existing three-line heading in the original order."""
    print("\n" + separator)
    print(title)
    print(separator)


Resilience input: /Users/wangjingjie/Desktop/nature skill/my paper/Energy Economics Code/matlab code/resilience_metrics.xlsx
Conditioning input: /Users/wangjingjie/Desktop/nature skill/my paper/Energy Economics Data/markov_scale_data_166.csv
Output directory: /Users/wangjingjie/Desktop/nature skill/my paper/Energy Economics Code/python code/output


In [2]:
%matplotlib inline

from datetime import datetime
from io import BytesIO

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import statsmodels.api as sm
from pathlib import Path



Matplotlib is building the font cache; this may take a moment.


In [3]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import norm
import statsmodels.api as sm

# 0) Basic settings

# Choose one uncertainty index:
#     "GPR" -> Var6  to Var10
#     "CPU" -> Var14 to Var18
#     "OPU" -> Var22 to Var26
#     "ALL" -> estimate and print all three indices
SELECTED_UNCERTAINTY = "ALL"

markets = ["OIL", "NG", "COAL", "ERIX", "EUA"]

uncertainty_variable_maps = {
    "GPR": {'OIL': 'Var6', 'NG': 'Var7', 'COAL': 'Var8', 'ERIX': 'Var9', 'EUA': 'Var10'},
    "CPU": {'OIL': 'Var14', 'NG': 'Var15', 'COAL': 'Var16', 'ERIX': 'Var17', 'EUA': 'Var18'},
    "OPU": {'OIL': 'Var22', 'NG': 'Var23', 'COAL': 'Var24', 'ERIX': 'Var25', 'EUA': 'Var26'}
}

exog_cols = ["DXY", "GSCPI", "CPI", "Industrial"]

# False: only print the paper-style RI and RD result tables.
# True: also print the complete statsmodels summary for every model.
PRINT_STATSMODELS_SUMMARY = False

y_path = RESILIENCE_FILE

x_path = MACRO_DATA_FILE

# Shared model utilities are defined in Cell 1.

# 2) Read data

r_data = pd.read_excel(y_path)
x_data = pd.read_csv(x_path)

missing_exog_cols = [column
    for column in exog_cols
    if column not in x_data.columns]

if missing_exog_cols:
    raise ValueError(f"The explanatory-variable file is missing columns: "
        f"{missing_exog_cols}")

# Do not drop missing observations here. Missing y and X values are
# removed jointly within each market model so the observations remain aligned.
Z_source = (x_data[exog_cols]
    .apply(pd.to_numeric, errors="coerce")
    .reset_index(drop=True))

selected_uncertainties = validate_uncertainty_selection(SELECTED_UNCERTAINTY)

# 3) Result containers

# Complete statsmodels results:
# results[uncertainty][measure][market]
results = {}

# Paper-style tables:
# paper_tables[uncertainty][measure]
paper_tables = {}

display_columns = paper_display_columns()

# 4) Estimate selected uncertainty index / indices

for uncertainty_name in selected_uncertainties:

    variable_map = uncertainty_variable_maps[uncertainty_name]
    validate_variable_codes(r_data=r_data, uncertainty_name=uncertainty_name,
        variable_map=variable_map)

    # Build RI and RD data for the selected uncertainty index.
    df_RI = build_dependent_dataframe(r_data=r_data, value_column="RP", variable_map=variable_map)

    df_RD = build_dependent_dataframe(r_data=r_data, value_column="RD", variable_map=variable_map)

    # Align the overall lengths for this uncertainty index.
    n = min(len(df_RI), len(df_RD), len(Z_source))

    if n == 0:
        raise ValueError(f"No observations are available for {uncertainty_name}.")

    df_RI = df_RI.iloc[:n].reset_index(drop=True)
    df_RD = df_RD.iloc[:n].reset_index(drop=True)
    Z_full = Z_source.iloc[:n].reset_index(drop=True)

    datasets = {"RI": df_RI, "RD": df_RD}

    results[uncertainty_name] = {"RI": {}, "RD": {}}

    paper_tables[uncertainty_name] = {}

    print_section(f'UNCERTAINTY INDEX: {uncertainty_name}', '#' * 120)

    for measure_name, dependent_data in datasets.items():

        print_section(f'Estimating {uncertainty_name} - {measure_name}', '#' * 100)

        paper_rows = []

        for market in markets:

            print_section(f'{uncertainty_name} - {measure_name} - {market}', '=' * 90)

            # 4.1 Jointly align y and X

            model_data = pd.concat([dependent_data[market].rename("y"), Z_full], axis=1)

            model_data = (model_data
                .dropna(subset=["y"] + exog_cols)
                .reset_index(drop=True))

            if model_data.empty:
                raise ValueError(f"No valid observations for "
                    f"{uncertainty_name} - {measure_name} - {market}.")

            y_original = model_data["y"].astype(float)
            Z = model_data[exog_cols].astype(float)

            # 4.2 Center dependent variable

            y_mean = y_original.mean()
            y_centered = y_original - y_mean

            # 4.3 Estimate Markov-switching regression

            model = sm.tsa.MarkovRegression(endog=y_centered, exog=Z, k_regimes=2, trend="c",
                switching_trend=True, switching_exog=True, switching_variance=True)

            result = model.fit(method="nm", maxiter=10000, disp=False)

            results[uncertainty_name][measure_name][market] = result

            if PRINT_STATSMODELS_SUMMARY:
                print(result.summary())

            # 4.4 Obtain named estimates and standard errors

            parameter_names = get_parameter_names(result)

            params = make_named_series(result.params, parameter_names)

            standard_errors = make_named_series(result.bse, parameter_names)

            # 4.5 Restore regime means to the original scale

            regime_results = {}

            for raw_regime in [0, 1]:

                const_name = find_regime_parameter(names=parameter_names, variable="const",
                    regime=raw_regime, exog_columns=exog_cols)

                centered_const = params[const_name]
                const_se = standard_errors[const_name]

                conditional_mean = y_mean + centered_const

                mean_z, mean_p = calculate_z_p(conditional_mean, const_se)

                regime_results[raw_regime] = {"conditional_mean": conditional_mean,
                    "conditional_mean_se": const_se, "conditional_mean_z": mean_z,
                    "conditional_mean_p": mean_p, "coefficients": {}}

                for variable in exog_cols:

                    coefficient_name = find_regime_parameter(names=parameter_names,
                        variable=variable, regime=raw_regime, exog_columns=exog_cols)

                    estimate = params[coefficient_name]
                    standard_error = standard_errors[coefficient_name]

                    z_value, p_value = calculate_z_p(estimate, standard_error)

                    regime_results[raw_regime]["coefficients"][variable] = {"estimate": estimate,
                        "standard_error": standard_error, "z": z_value, "p": p_value}

            # 4.6 Extract raw-regime persistence probabilities

            p00_name = find_parameter(parameter_names, "p[0->0]")

            p10_name = find_parameter(parameter_names, "p[1->0]")

            if p00_name is None or p10_name is None:
                raise ValueError(f"Cannot find transition parameters for "
                    f"{uncertainty_name} - {measure_name} - {market}.\n"
                    f"Available parameters:\n{parameter_names}")

            raw_persistence = {0: {"estimate": params[p00_name],
                    "standard_error": standard_errors[p00_name]}, 1: {
                    "estimate": 1.0 - params[p10_name], "standard_error": standard_errors[p10_name]
                }}

            # 4.7 Relabel regimes by economic interpretation

            state_1_raw, state_2_raw = relabel_regimes(measure_name=measure_name,
                regime_results=regime_results)

            # 4.8 Build one paper-style output row

            row = {"Market": market}

            state_1_result = regime_results[state_1_raw]
            state_2_result = regime_results[state_2_raw]

            row["State 1 Cond. mean"] = paper_format(state_1_result["conditional_mean"],
                state_1_result["conditional_mean_z"], state_1_result["conditional_mean_p"])

            for variable in exog_cols:
                coefficient = state_1_result["coefficients"][variable]

                row[f"State 1 {variable}"] = paper_format(coefficient["estimate"],
                    coefficient["z"], coefficient["p"])

            row["State 2 Cond. mean"] = paper_format(state_2_result["conditional_mean"],
                state_2_result["conditional_mean_z"], state_2_result["conditional_mean_p"])

            for variable in exog_cols:
                coefficient = state_2_result["coefficients"][variable]

                row[f"State 2 {variable}"] = paper_format(coefficient["estimate"],
                    coefficient["z"], coefficient["p"])

            # p11: probability of remaining in economically defined State 1
            p11 = raw_persistence[state_1_raw]["estimate"]
            p11_se = raw_persistence[state_1_raw]["standard_error"]
            p11_z, p11_p = calculate_z_p(p11, p11_se)

            row["p11"] = paper_format(p11, p11_z, p11_p)

            # p22: probability of remaining in economically defined State 2
            p22 = raw_persistence[state_2_raw]["estimate"]
            p22_se = raw_persistence[state_2_raw]["standard_error"]
            p22_z, p22_p = calculate_z_p(p22, p22_se)

            row["p22"] = paper_format(p22, p22_z, p22_p)

            # Retain raw-regime mapping for verification, but do not display it
            # in the final paper-style tables.
            row["State 1 raw regime"] = state_1_raw
            row["State 2 raw regime"] = state_2_raw

            paper_rows.append(row)

            print(f"Original mean = {y_mean:.6f}; "
                f"State 1 corresponds to raw regime {state_1_raw}; "
                f"State 2 corresponds to raw regime {state_2_raw}.")

        paper_tables[uncertainty_name][measure_name] = pd.DataFrame(paper_rows)

    # 5) Print the selected index's RI and RD tables

    print_section(f'{uncertainty_name} - Panel A: Resilience intensity (RI)', '#' * 120)

    print(paper_tables[uncertainty_name]["RI"][display_columns]
        .to_string(index=False))

    print_section(f'{uncertainty_name} - Panel B: Resilience duration (RD)', '#' * 120)

    print(paper_tables[uncertainty_name]["RD"][display_columns]
        .to_string(index=False))



########################################################################################################################
UNCERTAINTY INDEX: GPR
########################################################################################################################

####################################################################################################
Estimating GPR - RI
####################################################################################################

GPR - RI - OIL
Original mean = 0.792064; State 1 corresponds to raw regime 0; State 2 corresponds to raw regime 1.

GPR - RI - NG
Original mean = 0.739010; State 1 corresponds to raw regime 0; State 2 corresponds to raw regime 1.

GPR - RI - COAL
Original mean = 0.753993; State 1 corresponds to raw regime 1; State 2 corresponds to raw regime 0.

GPR - RI - ERIX
Original mean = 0.760870; State 1 corresponds to raw regime 0; State 2 corresponds to raw regime 1.

GPR - RI - EUA
Original mean = 0.766725; Stat

In [4]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import norm
import statsmodels.api as sm

# 0) Basic settings

# Choose one uncertainty index:
#     "GPR" -> Var6  to Var10
#     "CPU" -> Var14 to Var18
#     "OPU" -> Var22 to Var26
#     "ALL" -> estimate all three indices and export one workbook per index
SELECTED_UNCERTAINTY = "ALL"

# Probability selected for the R plotting column `prob_selected`:
#     1 -> economically defined State 1
#     2 -> economically defined State 2
STATE_TO_PLOT = 1

# True  -> export smoothed marginal probabilities
# False -> export filtered marginal probabilities
USE_SMOOTHED_PROBABILITIES = True

# False: only print the paper-style RI and RD result tables.
# True: also print the complete statsmodels summary for every model.
PRINT_STATSMODELS_SUMMARY = False

# Whether to export the probability workbook(s).
EXPORT_PROBABILITY_XLSX = True

markets = ["OIL", "NG", "COAL", "ERIX", "EUA"]

uncertainty_variable_maps = {
    "GPR": {'OIL': 'Var6', 'NG': 'Var7', 'COAL': 'Var8', 'ERIX': 'Var9', 'EUA': 'Var10'},
    "CPU": {'OIL': 'Var14', 'NG': 'Var15', 'COAL': 'Var16', 'ERIX': 'Var17', 'EUA': 'Var18'},
    "OPU": {'OIL': 'Var22', 'NG': 'Var23', 'COAL': 'Var24', 'ERIX': 'Var25', 'EUA': 'Var26'}
}

exog_cols = ["DXY", "GSCPI", "CPI", "Industrial"]
DATE_COLUMN = "date"

# Input files
y_path = RESILIENCE_FILE

x_path = MACRO_DATA_FILE

# Probability workbooks are saved here.
OUTPUT_DIR = NOTEBOOK_OUTPUT_DIR

# Probability-specific helpers; shared model utilities are in Cell 1.


def validate_state_to_plot(state_to_plot):
    """Validate the reported-state choice used for `prob_selected`."""
    if state_to_plot not in (1, 2):
        raise ValueError("STATE_TO_PLOT must be 1 or 2.")


def extract_probability_array(probability_object, regime_index, nobs):
    """
    Extract one raw-regime marginal-probability series as a NumPy array.

    This handles both pandas DataFrames and NumPy arrays returned by
    different statsmodels versions.
    """
    if isinstance(probability_object, pd.DataFrame):
        if regime_index in probability_object.columns:
            values = probability_object[regime_index].to_numpy(dtype=float)
        elif str(regime_index) in probability_object.columns:
            values = probability_object[str(regime_index)].to_numpy(dtype=float)
        else:
            values = probability_object.iloc[:, regime_index].to_numpy(dtype=float)

    elif isinstance(probability_object, pd.Series):
        if regime_index != 0:
            raise ValueError("A one-dimensional probability Series cannot supply "
                f"regime {regime_index}.")
        values = probability_object.to_numpy(dtype=float)

    else:
        array = np.asarray(probability_object, dtype=float)

        if array.ndim != 2:
            raise ValueError(f"Unexpected probability array shape: {array.shape}.")

        if array.shape[0] == nobs:
            values = array[:, regime_index]
        elif array.shape[1] == nobs:
            values = array[regime_index, :]
        else:
            raise ValueError(f"Unexpected probability array shape: {array.shape}; "
                f"one dimension must equal nobs={nobs}.")

    if len(values) != nobs:
        raise ValueError(f"Probability series has length {len(values)}, "
            f"but the model sample contains {nobs} observations.")

    return np.asarray(values, dtype=float)


def build_probability_table(result, model_data, uncertainty_name, measure_name, market,
    state_1_raw, state_2_raw, state_to_plot, use_smoothed,):
    """Build the long-form probability output for one fitted model."""
    probability_object = (result.smoothed_marginal_probabilities
        if use_smoothed
        else result.filtered_marginal_probabilities)

    nobs = len(model_data)
    prob_raw_0 = extract_probability_array(probability_object=probability_object, regime_index=0,
        nobs=nobs)
    prob_raw_1 = extract_probability_array(probability_object=probability_object, regime_index=1,
        nobs=nobs)

    raw_probability_map = {0: prob_raw_0, 1: prob_raw_1}

    prob_state_1 = raw_probability_map[state_1_raw]
    prob_state_2 = raw_probability_map[state_2_raw]

    if state_to_plot == 1:
        selected_raw_regime = state_1_raw
        prob_selected = prob_state_1
    else:
        selected_raw_regime = state_2_raw
        prob_selected = prob_state_2

    probability_type = "smoothed" if use_smoothed else "filtered"

    return pd.DataFrame({"date": model_data[DATE_COLUMN].to_numpy(), "shock": uncertainty_name,
        "measure": measure_name, "market": market,
        "metric_value": model_data["y"].to_numpy(dtype=float),
        "y_centered": model_data["y_centered"].to_numpy(dtype=float),
        "probability_type": probability_type, "prob_raw_regime0": prob_raw_0,
        "prob_raw_regime1": prob_raw_1, "prob_state1": prob_state_1, "prob_state2": prob_state_2,
        "state_to_plot": state_to_plot, "selected_raw_regime": selected_raw_regime,
        "prob_selected": prob_selected})


def build_state_mapping_row(result, uncertainty_name, measure_name, market, y_mean, regime_results,
    state_1_raw, state_2_raw, state_to_plot, use_smoothed,):
    """Build one state-mapping record for checking R plot labels."""
    selected_raw_regime = (state_1_raw if state_to_plot == 1 else state_2_raw)

    if measure_name == "RI":
        state_1_definition = "higher original-scale conditional mean"
        state_2_definition = "lower original-scale conditional mean"
    else:
        state_1_definition = "lower original-scale conditional mean"
        state_2_definition = "higher original-scale conditional mean"

    mle_retvals = getattr(result, "mle_retvals", {})
    converged = (mle_retvals.get("converged", np.nan)
        if isinstance(mle_retvals, dict)
        else np.nan)

    return {"shock": uncertainty_name, "measure": measure_name, "market": market,
        "probability_type": "smoothed" if use_smoothed else "filtered",
        "state1_definition": state_1_definition, "state2_definition": state_2_definition,
        "state_to_plot": state_to_plot, "selected_raw_regime": selected_raw_regime,
        "y_mean": float(y_mean), "const0_centered": float(regime_results[0]["centered_const"]),
        "const1_centered": float(regime_results[1]["centered_const"]), "const0_original": float(
            regime_results[0]["conditional_mean"]), "const1_original": float(
            regime_results[1]["conditional_mean"]), "state1_raw": int(state_1_raw),
        "state2_raw": int(state_2_raw), "state1_original_intercept": float(
            regime_results[state_1_raw]["conditional_mean"]), "state2_original_intercept": float(
            regime_results[state_2_raw]["conditional_mean"]), "nobs": int(result.nobs),
        "converged": converged}


def export_probability_workbook(uncertainty_name, probability_table, state_mapping_table,
    output_dir, state_to_plot,):
    """Export R-ready probabilities and state mapping to one XLSX file."""
    output_dir.mkdir(parents=True, exist_ok=True)

    output_path = output_dir / (f"{uncertainty_name.lower()}_ms_state_probabilities_"
        f"state{state_to_plot}.xlsx")

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        probability_table.to_excel(writer, sheet_name="probabilities_long", index=False)
        state_mapping_table.to_excel(writer, sheet_name="state_mapping", index=False)

        # Apply basic formats without adding extra analytical outputs.
        workbook = writer.book

        probability_sheet = writer.sheets["probabilities_long"]
        probability_sheet.freeze_panes = "A2"
        probability_sheet.auto_filter.ref = probability_sheet.dimensions
        probability_sheet.column_dimensions["A"].width = 13

        for cell in probability_sheet["A"][1:]:
            cell.number_format = "yyyy-mm-dd"

        mapping_sheet = writer.sheets["state_mapping"]
        mapping_sheet.freeze_panes = "A2"
        mapping_sheet.auto_filter.ref = mapping_sheet.dimensions

        # Keep all worksheets visible and leave the workbook active on
        # probabilities_long, which is the sheet used directly in R.
        workbook.active = workbook.sheetnames.index("probabilities_long")

    print(f"Probability workbook saved to: {output_path}")
    return output_path

# 2) Read and validate data

if not y_path.exists():
    raise FileNotFoundError(f"Dependent-variable file not found: {y_path}")

if not x_path.exists():
    raise FileNotFoundError(f"Explanatory-variable file not found: {x_path}")

r_data = pd.read_excel(y_path)
x_data = pd.read_csv(x_path)

required_y_columns = {"Variable", "RP", "RD"}
missing_y_columns = sorted(required_y_columns - set(r_data.columns))

if missing_y_columns:
    raise ValueError(f"resilience_metrics.xlsx is missing columns: {missing_y_columns}")

missing_exog_cols = [column
    for column in exog_cols
    if column not in x_data.columns]

if missing_exog_cols:
    raise ValueError(f"The explanatory-variable file is missing columns: "
        f"{missing_exog_cols}")

if DATE_COLUMN not in x_data.columns:
    raise ValueError(f"The explanatory-variable file must contain a '{DATE_COLUMN}' "
        "column so the exported probabilities can be plotted in R.")

# Do not drop missing observations here. Missing y, X, and date values are
# removed jointly within each market model so observations remain aligned.
Z_source = (x_data[exog_cols]
    .apply(pd.to_numeric, errors="coerce")
    .reset_index(drop=True))

date_source = (pd.to_datetime(x_data[DATE_COLUMN], errors="coerce", utc=True)
    .dt.tz_convert(None)
    .reset_index(drop=True))

selected_uncertainties = validate_uncertainty_selection(SELECTED_UNCERTAINTY)
validate_state_to_plot(STATE_TO_PLOT)

# 3) Result containers

# Complete statsmodels results:
# results[uncertainty][measure][market]
results = {}

# Paper-style tables:
# paper_tables[uncertainty][measure]
paper_tables = {}

# Long probability outputs:
# probability_tables[uncertainty]
probability_tables = {}

# State-label checks:
# state_mapping_tables[uncertainty]
state_mapping_tables = {}

# Exported workbook paths:
# exported_probability_files[uncertainty]
exported_probability_files = {}

display_columns = paper_display_columns()

# 4) Estimate selected uncertainty index / indices

for uncertainty_name in selected_uncertainties:

    variable_map = uncertainty_variable_maps[uncertainty_name]
    validate_variable_codes(r_data=r_data, uncertainty_name=uncertainty_name,
        variable_map=variable_map)

    # Build RI and RD data for this uncertainty index.
    df_RI = build_dependent_dataframe(r_data=r_data, value_column="RP", variable_map=variable_map)

    df_RD = build_dependent_dataframe(r_data=r_data, value_column="RD", variable_map=variable_map)

    # Align the overall positional lengths for this uncertainty index.
    n = min(len(df_RI), len(df_RD), len(Z_source), len(date_source))

    if n == 0:
        raise ValueError(f"No observations are available for {uncertainty_name}.")

    df_RI = df_RI.iloc[:n].reset_index(drop=True)
    df_RD = df_RD.iloc[:n].reset_index(drop=True)
    Z_full = Z_source.iloc[:n].reset_index(drop=True)
    dates_full = date_source.iloc[:n].reset_index(drop=True)

    datasets = {"RI": df_RI, "RD": df_RD}

    results[uncertainty_name] = {"RI": {}, "RD": {}}

    paper_tables[uncertainty_name] = {}

    uncertainty_probability_parts = []
    uncertainty_mapping_rows = []

    print_section(f'UNCERTAINTY INDEX: {uncertainty_name}', '#' * 120)

    for measure_name, dependent_data in datasets.items():

        print_section(f'Estimating {uncertainty_name} - {measure_name}', '#' * 100)

        paper_rows = []

        for market in markets:

            print_section(f'{uncertainty_name} - {measure_name} - {market}', '=' * 90)

            # 4.1 Jointly align date, y, and X

            model_data = pd.concat([dates_full.rename(DATE_COLUMN),
                    dependent_data[market].rename("y"), Z_full], axis=1)

            model_data = (model_data
                .dropna(subset=[DATE_COLUMN, "y"] + exog_cols)
                .sort_values(DATE_COLUMN)
                .reset_index(drop=True))

            if model_data.empty:
                raise ValueError(f"No valid observations for "
                    f"{uncertainty_name} - {measure_name} - {market}.")

            y_original = model_data["y"].astype(float)
            Z = model_data[exog_cols].astype(float)

            # 4.2 Center dependent variable

            y_mean = float(y_original.mean())
            y_centered = y_original - y_mean
            model_data["y_centered"] = y_centered.to_numpy(dtype=float)

            # 4.3 Estimate Markov-switching regression

            model = sm.tsa.MarkovRegression(endog=y_centered, exog=Z, k_regimes=2, trend="c",
                switching_trend=True, switching_exog=True, switching_variance=True)

            result = model.fit(method="nm", maxiter=10000, disp=False)

            results[uncertainty_name][measure_name][market] = result

            if PRINT_STATSMODELS_SUMMARY:
                print(result.summary())

            # 4.4 Obtain named estimates and standard errors

            parameter_names = get_parameter_names(result)

            params = make_named_series(result.params, parameter_names)

            standard_errors = make_named_series(result.bse, parameter_names)

            # 4.5 Restore regime means to the original scale

            regime_results = {}

            for raw_regime in [0, 1]:

                const_name = find_regime_parameter(names=parameter_names, variable="const",
                    regime=raw_regime, exog_columns=exog_cols)

                centered_const = float(params[const_name])
                const_se = float(standard_errors[const_name])
                conditional_mean = y_mean + centered_const

                mean_z, mean_p = calculate_z_p(conditional_mean, const_se)

                regime_results[raw_regime] = {"centered_const": centered_const,
                    "conditional_mean": conditional_mean, "conditional_mean_se": const_se,
                    "conditional_mean_z": mean_z, "conditional_mean_p": mean_p, "coefficients": {}}

                for variable in exog_cols:

                    coefficient_name = find_regime_parameter(names=parameter_names,
                        variable=variable, regime=raw_regime, exog_columns=exog_cols)

                    estimate = float(params[coefficient_name])
                    standard_error = float(standard_errors[coefficient_name])

                    z_value, p_value = calculate_z_p(estimate, standard_error)

                    regime_results[raw_regime]["coefficients"][variable] = {"estimate": estimate,
                        "standard_error": standard_error, "z": z_value, "p": p_value}

            # 4.6 Extract raw-regime persistence probabilities

            p00_name = find_parameter(parameter_names, "p[0->0]")

            p10_name = find_parameter(parameter_names, "p[1->0]")

            if p00_name is None or p10_name is None:
                raise ValueError(f"Cannot find transition parameters for "
                    f"{uncertainty_name} - {measure_name} - {market}.\n"
                    f"Available parameters:\n{parameter_names}")

            raw_persistence = {0: {"estimate": float(params[p00_name]),
                    "standard_error": float(standard_errors[p00_name])}, 1: {
                    "estimate": float(1.0 - params[p10_name]),
                    "standard_error": float(standard_errors[p10_name])}}

            # 4.7 Relabel regimes by economic interpretation

            state_1_raw, state_2_raw = relabel_regimes(measure_name=measure_name,
                regime_results=regime_results)

            # 4.8 Build one paper-style output row

            row = {"Market": market}

            state_1_result = regime_results[state_1_raw]
            state_2_result = regime_results[state_2_raw]

            row["State 1 Cond. mean"] = paper_format(state_1_result["conditional_mean"],
                state_1_result["conditional_mean_z"], state_1_result["conditional_mean_p"])

            for variable in exog_cols:
                coefficient = state_1_result["coefficients"][variable]

                row[f"State 1 {variable}"] = paper_format(coefficient["estimate"],
                    coefficient["z"], coefficient["p"])

            row["State 2 Cond. mean"] = paper_format(state_2_result["conditional_mean"],
                state_2_result["conditional_mean_z"], state_2_result["conditional_mean_p"])

            for variable in exog_cols:
                coefficient = state_2_result["coefficients"][variable]

                row[f"State 2 {variable}"] = paper_format(coefficient["estimate"],
                    coefficient["z"], coefficient["p"])

            # p11: probability of remaining in economically defined State 1
            p11 = raw_persistence[state_1_raw]["estimate"]
            p11_se = raw_persistence[state_1_raw]["standard_error"]
            p11_z, p11_p = calculate_z_p(p11, p11_se)

            row["p11"] = paper_format(p11, p11_z, p11_p)

            # p22: probability of remaining in economically defined State 2
            p22 = raw_persistence[state_2_raw]["estimate"]
            p22_se = raw_persistence[state_2_raw]["standard_error"]
            p22_z, p22_p = calculate_z_p(p22, p22_se)

            row["p22"] = paper_format(p22, p22_z, p22_p)

            # Retain raw-regime mapping for verification, but do not display it
            # in the final paper-style tables.
            row["State 1 raw regime"] = state_1_raw
            row["State 2 raw regime"] = state_2_raw

            paper_rows.append(row)

            # 4.9 Reuse this fitted result to export probabilities

            probability_part = build_probability_table(result=result, model_data=model_data,
                uncertainty_name=uncertainty_name, measure_name=measure_name, market=market,
                state_1_raw=state_1_raw, state_2_raw=state_2_raw, state_to_plot=STATE_TO_PLOT,
                use_smoothed=USE_SMOOTHED_PROBABILITIES)
            uncertainty_probability_parts.append(probability_part)

            mapping_row = build_state_mapping_row(result=result, uncertainty_name=uncertainty_name,
                measure_name=measure_name, market=market, y_mean=y_mean,
                regime_results=regime_results, state_1_raw=state_1_raw, state_2_raw=state_2_raw,
                state_to_plot=STATE_TO_PLOT, use_smoothed=USE_SMOOTHED_PROBABILITIES)
            uncertainty_mapping_rows.append(mapping_row)

            selected_raw_regime = (state_1_raw if STATE_TO_PLOT == 1 else state_2_raw)

            print(f"Original mean = {y_mean:.6f}; "
                f"State 1 corresponds to raw regime {state_1_raw}; "
                f"State 2 corresponds to raw regime {state_2_raw}; "
                f"prob_selected uses raw regime {selected_raw_regime}.")

        paper_tables[uncertainty_name][measure_name] = pd.DataFrame(paper_rows)

    # 5) Print this index's RI and RD result tables

    print_section(f'{uncertainty_name} - Panel A: Resilience intensity (RI)', '#' * 120)

    print(paper_tables[uncertainty_name]["RI"][display_columns]
        .to_string(index=False))

    print_section(f'{uncertainty_name} - Panel B: Resilience duration (RD)', '#' * 120)

    print(paper_tables[uncertainty_name]["RD"][display_columns]
        .to_string(index=False))

    # 6) Combine and export probability outputs

    # Preserve the same row order as the original probability-export code:
    # OIL-RI, OIL-RD, NG-RI, NG-RD, ..., EUA-RI, EUA-RD.
    # Do not sort market names alphabetically.
    market_order = {market: position for position, market in enumerate(markets)}
    measure_order = {"RI": 0, "RD": 1}

    probability_tables[uncertainty_name] = pd.concat(uncertainty_probability_parts,
        ignore_index=True)

    probability_tables[uncertainty_name]["_market_order"] = (
        probability_tables[uncertainty_name]["market"].map(market_order))
    probability_tables[uncertainty_name]["_measure_order"] = (
        probability_tables[uncertainty_name]["measure"].map(measure_order))

    probability_tables[uncertainty_name] = (probability_tables[uncertainty_name]
        .sort_values(["_market_order", "_measure_order", "date"], kind="stable")
        .drop(columns=["_market_order", "_measure_order"])
        .reset_index(drop=True))

    state_mapping_tables[uncertainty_name] = pd.DataFrame(uncertainty_mapping_rows)

    state_mapping_tables[uncertainty_name]["_market_order"] = (
        state_mapping_tables[uncertainty_name]["market"].map(market_order))
    state_mapping_tables[uncertainty_name]["_measure_order"] = (
        state_mapping_tables[uncertainty_name]["measure"].map(measure_order))

    state_mapping_tables[uncertainty_name] = (state_mapping_tables[uncertainty_name]
        .sort_values(["_market_order", "_measure_order"], kind="stable")
        .drop(columns=["_market_order", "_measure_order"])
        .reset_index(drop=True))

    if EXPORT_PROBABILITY_XLSX:
        exported_probability_files[uncertainty_name] = (export_probability_workbook(
                uncertainty_name=uncertainty_name,
                probability_table=probability_tables[uncertainty_name],
                state_mapping_table=state_mapping_tables[uncertainty_name], output_dir=OUTPUT_DIR,
                state_to_plot=STATE_TO_PLOT))



########################################################################################################################
UNCERTAINTY INDEX: GPR
########################################################################################################################

####################################################################################################
Estimating GPR - RI
####################################################################################################

GPR - RI - OIL
Original mean = 0.792064; State 1 corresponds to raw regime 0; State 2 corresponds to raw regime 1; prob_selected uses raw regime 0.

GPR - RI - NG
Original mean = 0.739010; State 1 corresponds to raw regime 0; State 2 corresponds to raw regime 1; prob_selected uses raw regime 0.

GPR - RI - COAL
Original mean = 0.753993; State 1 corresponds to raw regime 1; State 2 corresponds to raw regime 0; prob_selected uses raw regime 1.

GPR - RI - ERIX
Original mean = 0.760870; State 1 corresponds t

In [5]:
# Individual cross-state slope equality: full-covariance Wald tests.
# Run after the probability-export estimation cell; preserve economic state labels.
import numpy as np
import pandas as pd
from scipy.stats import chi2

EXPORT_EQUALITY_XLSX = True
EQUALITY_OUTPUT_PATH = OUTPUT_DIR / "cross_state_equality_tests.xlsx"


def find_regime_coef_index(names, variable, regime):
    """Locate the parameter index of one regime coefficient.

    Supports both statsmodels naming formats:
        DXY[0], GSCPI[1], ...
    and:
        x1[0], x2[1], ...
    """
    if variable == "const":
        candidates = [f"const[{regime}]"]
    else:
        candidates = [f"{variable}[{regime}]"]
        position = exog_cols.index(variable) + 1
        candidates.append(f"x{position}[{regime}]")
    for i, nm in enumerate(names):
        if str(nm) in candidates:
            return i
    return None


def significance_stars_eq(p_value):
    if pd.isna(p_value):
        return ""
    if p_value < 0.01:
        return "***"
    if p_value < 0.05:
        return "**"
    if p_value < 0.10:
        return "*"
    return ""

equality_rows = []

for uncertainty_name, shock_results in results.items():
    for measure_name, market_results in shock_results.items():
        for market, result in market_results.items():

            names = list(result.params.index) if hasattr(result.params, "index") else list(result.model.param_names)
            params_vec = np.asarray(result.params, dtype=float)
            cov = np.asarray(result.cov_params(), dtype=float)

            # Identify the raw regimes corresponding to State 1 and State 2
            # under the economic labeling used in the paper.
            const_idx = {r: find_regime_coef_index(names, "const", r) for r in (0, 1)}
            if const_idx[0] is None or const_idx[1] is None:
                continue
            cond_means = {r: params_vec[const_idx[r]] for r in (0, 1)}
            if measure_name == "RI":
                state1_raw = max((0, 1), key=lambda r: cond_means[r])
            else:
                state1_raw = min((0, 1), key=lambda r: cond_means[r])
            state2_raw = 1 - state1_raw

            for variable in exog_cols:
                i1 = find_regime_coef_index(names, variable, state1_raw)
                i2 = find_regime_coef_index(names, variable, state2_raw)
                if i1 is None or i2 is None:
                    continue

                beta1 = params_vec[i1]
                beta2 = params_vec[i2]

                # Wald test of beta_{s1} - beta_{s2} = 0.
                R = np.zeros((1, len(params_vec)))
                R[0, i1] = 1.0
                R[0, i2] = -1.0
                diff = (R @ params_vec).item()
                var = (R @ cov @ R.T).item()

                if var <= 0 or not np.isfinite(var):
                    wald = np.nan
                    p_value = np.nan
                else:
                    wald = diff ** 2 / var
                    p_value = float(chi2.sf(wald, df=1))

                equality_rows.append({"shock": uncertainty_name, "measure": measure_name,
                    "market": market, "variable": variable, "beta_state1": round(beta1, 4),
                    "beta_state2": round(beta2, 4), "sign_opposite": (beta1 * beta2) < 0,
                    "diff": round(diff, 4), "se_diff": round(var ** 0.5, 4),
                    "wald": round(wald, 3) if pd.notna(wald) else np.nan, "p_value": p_value,
                    "stars": significance_stars_eq(p_value)})

equality_table = pd.DataFrame(equality_rows)

print("#" * 110)
print("CROSS-STATE COEFFICIENT EQUALITY TESTS   H0: beta1 = beta2 (Wald, chi2, df=1)")
print("State 1 / State 2 follow the economic labels: RI State1 = high absorption, RD State1 = fast recovery")
print("#" * 110)
print(equality_table[["shock", "measure", "market", "variable",
                    "beta_state1", "beta_state2", "sign_opposite",
                    "diff", "se_diff", "wald", "p_value", "stars"]]
    .to_string(index=False))

# Summary counts: among sign-opposite cases, how many are significant?
opp = equality_table[equality_table["sign_opposite"]]
print("\nSign-opposite cases: {} of {} tests".format(len(opp), len(equality_table)))
print("Sign-opposite AND significant at 10%: {}".format((opp["p_value"] < 0.10).sum()))
print("Sign-opposite AND significant at 5%: {}".format((opp["p_value"] < 0.05).sum()))
print("Sign-opposite AND significant at 1%: {}".format((opp["p_value"] < 0.01).sum()))

if EXPORT_EQUALITY_XLSX:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(EQUALITY_OUTPUT_PATH, engine="openpyxl") as writer:
        equality_table.to_excel(writer, sheet_name="equality_tests", index=False)
        sheet = writer.sheets["equality_tests"]
        sheet.freeze_panes = "A2"
        sheet.auto_filter.ref = sheet.dimensions
    print(f"\nEquality-test workbook saved to: {EQUALITY_OUTPUT_PATH}")


##############################################################################################################
CROSS-STATE COEFFICIENT EQUALITY TESTS   H0: beta1 = beta2 (Wald, chi2, df=1)
State 1 / State 2 follow the economic labels: RI State1 = high absorption, RD State1 = fast recovery
##############################################################################################################
shock measure market   variable  beta_state1  beta_state2  sign_opposite    diff  se_diff   wald      p_value stars
  GPR      RI    OIL        DXY      -0.0031       0.0119           True -0.0150   0.0094  2.549 1.103864e-01      
  GPR      RI    OIL      GSCPI       0.0005       0.0218          False -0.0212   0.0095  5.023 2.501573e-02    **
  GPR      RI    OIL        CPI      -0.0038      -0.0143          False  0.0105   0.0088  1.423 2.328631e-01      
  GPR      RI    OIL Industrial      -0.0006       0.0305           True -0.0310   0.0308  1.016 3.133528e-01      
  GPR      RI     N

In [6]:
# Comprehensive level, individual-slope, and joint-slope Wald tests.
# Individual level and slope tests share the original pooled BH correction.
import numpy as np
import pandas as pd
from scipy.stats import chi2
from statsmodels.stats.multitest import multipletests

EXPORT_COMPREHENSIVE_TESTS_XLSX = True

# Keep the user's existing output convention when OUTPUT_DIR is available.
# Otherwise write to the current working directory.
try:
    _test_output_dir = Path(OUTPUT_DIR)
except NameError:
    _test_output_dir = Path.cwd()

COMPREHENSIVE_TEST_OUTPUT_PATH = (_test_output_dir / "cross_state_comprehensive_tests.xlsx")


def _find_regime_parameter_index(names, variable, regime):
    """Return the parameter index for a regime-specific intercept or slope."""
    if variable == "const":
        candidates = [f"const[{regime}]"]
    else:
        candidates = [f"{variable}[{regime}]"]
        position = exog_cols.index(variable) + 1
        candidates.append(f"x{position}[{regime}]")

    for i, name in enumerate(names):
        if str(name) in candidates:
            return i
    return None


def _wald_linear_restriction(params, cov, R):
    """Wald test for H0: R @ params = 0."""
    params = np.asarray(params, dtype=float)
    cov = np.asarray(cov, dtype=float)
    R = np.atleast_2d(np.asarray(R, dtype=float))

    restriction = R @ params
    middle = R @ cov @ R.T

    if not (np.all(np.isfinite(restriction))
        and np.all(np.isfinite(middle))):
        return np.nan, np.nan, np.full(len(restriction), np.nan), len(restriction)

    # pinv is preferable here to an unconditional inverse because estimated
    # covariance matrices can be numerically close to singular.
    middle_inv = np.linalg.pinv(middle)
    statistic = float(restriction.T @ middle_inv @ restriction)
    df = int(np.linalg.matrix_rank(R))

    if df <= 0 or statistic < 0 or not np.isfinite(statistic):
        return np.nan, np.nan, restriction, df

    p_value = float(chi2.sf(statistic, df=df))
    return statistic, p_value, restriction, df


def _stars(p_value):
    if pd.isna(p_value):
        return ""
    if p_value <= 0.01:
        return "***"
    if p_value <= 0.05:
        return "**"
    if p_value <= 0.10:
        return "*"
    return ""

individual_rows = []
joint_rows = []

for uncertainty_name, shock_results in results.items():
    for measure_name, market_results in shock_results.items():
        for market, result in market_results.items():

            names = (list(result.params.index)
                if hasattr(result.params, "index")
                else list(result.model.param_names))
            params = np.asarray(result.params, dtype=float)
            cov = np.asarray(result.cov_params(), dtype=float)

            # A) Recover the economic State 1 / State 2 labels
            const_idx = {r: _find_regime_parameter_index(names, "const", r)
                for r in (0, 1)}
            if const_idx[0] is None or const_idx[1] is None:
                raise ValueError(f"Cannot find both regime intercepts for "
                    f"{uncertainty_name}-{measure_name}-{market}.")

            raw_const = {r: params[const_idx[r]] for r in (0, 1)}

            if measure_name == "RI":
                state1_raw = max((0, 1), key=lambda r: raw_const[r])
            elif measure_name == "RD":
                state1_raw = min((0, 1), key=lambda r: raw_const[r])
            else:
                raise ValueError(f"Unknown measure: {measure_name}")

            state2_raw = 1 - state1_raw

            # B) Intercept / conditional-level equality
            i1 = const_idx[state1_raw]
            i2 = const_idx[state2_raw]

            R_const = np.zeros((1, len(params)))
            R_const[0, i1] = 1.0
            R_const[0, i2] = -1.0

            wald, p_value, restriction, df = _wald_linear_restriction(params, cov, R_const)
            diff = float(restriction[0]) if np.all(np.isfinite(restriction)) else np.nan
            var_diff = float((R_const @ cov @ R_const.T).item())
            se_diff = np.sqrt(var_diff) if var_diff > 0 else np.nan

            individual_rows.append({"shock": uncertainty_name, "measure": measure_name,
                "market": market, "test_type": "intercept", "variable": "const",
                "state1_raw_regime": state1_raw, "state2_raw_regime": state2_raw,
                "estimate_state1": params[i1], "estimate_state2": params[i2],
                "difference_state1_minus_state2": diff, "se_difference": se_diff, "wald_chi2": wald,
                "df": df, "p_value": p_value})

            # C) Variable-specific slope equality tests
            slope_restrictions = []

            for variable in exog_cols:
                j1 = _find_regime_parameter_index(names, variable, state1_raw)
                j2 = _find_regime_parameter_index(names, variable, state2_raw)

                if j1 is None or j2 is None:
                    raise ValueError(f"Cannot find {variable} coefficients for "
                        f"{uncertainty_name}-{measure_name}-{market}.")

                R_var = np.zeros((1, len(params)))
                R_var[0, j1] = 1.0
                R_var[0, j2] = -1.0
                slope_restrictions.append(R_var[0])

                wald, p_value, restriction, df = _wald_linear_restriction(params, cov, R_var)
                diff = (float(restriction[0])
                    if np.all(np.isfinite(restriction))
                    else np.nan)
                var_diff = float((R_var @ cov @ R_var.T).item())
                se_diff = np.sqrt(var_diff) if var_diff > 0 else np.nan

                beta1 = float(params[j1])
                beta2 = float(params[j2])

                individual_rows.append({"shock": uncertainty_name, "measure": measure_name,
                    "market": market, "test_type": "slope", "variable": variable,
                    "state1_raw_regime": state1_raw, "state2_raw_regime": state2_raw,
                    "estimate_state1": beta1, "estimate_state2": beta2,
                    "difference_state1_minus_state2": diff, "se_difference": se_diff,
                    "sign_opposite": (beta1 * beta2) < 0, "wald_chi2": wald, "df": df,
                    "p_value": p_value})

            # D) Joint equality of all four macrofinancial slopes
            R_joint = np.vstack(slope_restrictions)
            wald_joint, p_joint, _, df_joint = _wald_linear_restriction(params, cov, R_joint)

            joint_rows.append({"shock": uncertainty_name, "measure": measure_name,
                "market": market, "test": "all four slopes equal across states",
                "wald_chi2": wald_joint, "df": df_joint, "p_value": p_joint})

individual_tests = pd.DataFrame(individual_rows)
joint_tests = pd.DataFrame(joint_rows)

# E) Benjamini-Hochberg FDR correction
# Apply FDR to the full family of individual cross-state tests. This leaves the
# original p-values untouched and adds q-values for robustness to multiplicity.
valid = individual_tests["p_value"].notna()
individual_tests["q_value_bh"] = np.nan
individual_tests.loc[valid, "q_value_bh"] = multipletests(
    individual_tests.loc[valid, "p_value"].to_numpy(), alpha=0.05, method="fdr_bh")[1]

individual_tests["stars_p"] = individual_tests["p_value"].apply(_stars)
individual_tests["stars_q"] = individual_tests["q_value_bh"].apply(_stars)
joint_tests["stars_p"] = joint_tests["p_value"].apply(_stars)

# More readable ordering
individual_tests = individual_tests.sort_values(
    ["shock", "measure", "market", "test_type", "variable"]).reset_index(drop=True)
joint_tests = joint_tests.sort_values(["shock", "measure", "market"]).reset_index(drop=True)

# F) Compact diagnostic summaries
intercept_tests = individual_tests[individual_tests["test_type"] == "intercept"].copy()
slope_tests = individual_tests[individual_tests["test_type"] == "slope"].copy()

print("#" * 118)
print("COMPREHENSIVE CROSS-STATE WALD TESTS")
print("RI State 1 = higher absorption; RD State 1 = responses concentrated earlier")
print("#" * 118)

print("\n[1] Intercept / regime-level equality: H0: alpha_1 = alpha_2")
print(intercept_tests[["shock", "measure", "market", "estimate_state1", "estimate_state2",
        "difference_state1_minus_state2", "se_difference",
        "wald_chi2", "p_value", "q_value_bh", "stars_p", "stars_q"]].to_string(index=False))

print("\n[2] Variable-specific slope equality: H0: beta_{k,1} = beta_{k,2}")
print(slope_tests[["shock", "measure", "market", "variable",
        "estimate_state1", "estimate_state2", "sign_opposite",
        "difference_state1_minus_state2", "se_difference",
        "wald_chi2", "p_value", "q_value_bh", "stars_p", "stars_q"]].to_string(index=False))

print("\n[3] Joint slope equality: H0: beta_1 = beta_2 for DXY, GSCPI, CPI, Industrial")
print(joint_tests[["shock", "measure", "market", "wald_chi2", "df", "p_value", "stars_p"
    ]].to_string(index=False))

print("\nSummary")
print(f"Individual intercept tests: {len(intercept_tests)}")
print(f"Individual slope tests:     {len(slope_tests)}")
print(f"Joint slope tests:          {len(joint_tests)}")
print("Individual tests significant at 5%: "
    f"raw p = {(individual_tests['p_value'] < 0.05).sum()}, "
    f"BH q = {(individual_tests['q_value_bh'] < 0.05).sum()}")
print("Joint slope tests significant at 5%: "
    f"{(joint_tests['p_value'] < 0.05).sum()}")

# G) Export
if EXPORT_COMPREHENSIVE_TESTS_XLSX:
    _test_output_dir.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(COMPREHENSIVE_TEST_OUTPUT_PATH, engine="openpyxl") as writer:
        individual_tests.to_excel(writer, sheet_name="individual_tests", index=False)
        joint_tests.to_excel(writer, sheet_name="joint_slope_tests", index=False)

        for sheet_name in ["individual_tests", "joint_slope_tests"]:
            ws = writer.sheets[sheet_name]
            ws.freeze_panes = "A2"
            ws.auto_filter.ref = ws.dimensions

    print(f"\nComprehensive test workbook saved to: {COMPREHENSIVE_TEST_OUTPUT_PATH}")


######################################################################################################################
COMPREHENSIVE CROSS-STATE WALD TESTS
RI State 1 = higher absorption; RD State 1 = responses concentrated earlier
######################################################################################################################

[1] Intercept / regime-level equality: H0: alpha_1 = alpha_2
shock measure market  estimate_state1  estimate_state2  difference_state1_minus_state2  se_difference  wald_chi2       p_value    q_value_bh stars_p stars_q
  CPU      RD   COAL        -0.358660         0.185507                       -0.544167       0.053473 103.561498  2.524356e-24  2.704667e-23     ***     ***
  CPU      RD   ERIX        -0.036941         0.153105                       -0.190046       0.060161   9.979098  1.583272e-03  7.916361e-03     ***     ***
  CPU      RD    EUA        -0.164418         0.134468                       -0.298887       0.036443  67.263818  2.